<a href="https://colab.research.google.com/github/varadasantosh/deep-learning-notes/blob/tensorflow/transformers/transformer_model_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn as nn

# Input Embeddings

In [ ]:
class InputEmbeddings(nn.Module):

  def __init__(self, d_model, vocab_size):
    super().__init__()
    self.d_model = d_model
    self.vocab_size = vocab_size
    self.embedding_layer= nn.Embedding(vocab_size, d_model)

  def forward(self, x):
    return self.embedding_layer(x)


# Positional Embeddings

In [ ]:
class PositionalEmbeddings(nn.Module):

  def __init__(self, seq_len,d_model):

    super().__init__()
    token_positions = torch.arange(0,seq_len, dtype=torch.int64)
    token_positions = token_positions.unsqueeze(1)
    position_embeddings = torch.zeros(seq_len, d_model)
    denominator = torch.pow(10000, 2*torch.arange(0,d_model,2)/(d_model**0.5))
    position_embeddings[:,0::2] = torch.sin(token_positions/denominator)
    position_embeddings[:,1::2] = torch.cos(token_positions/denominator)
    self.register_buffer('position_embeddings', position_embeddings.unsqueeze(0))

  def forward(self,x):
     return x + (self.positional_embeddings[:, :x.shape[1], :]).requires_grad_(False)


In [ ]:
def position_embeddings(seq_len, d_model):
    token_positions = torch.arange(0,seq_len, dtype=torch.int64)
    token_positions = token_positions.unsqueeze(1)
    position_embeddings = torch.zeros(seq_len, d_model)
    denominator = torch.pow(10000, 2*torch.arange(0,d_model,2)/(d_model**0.5))
    position_embeddings[:,0::2] = torch.sin(token_positions/denominator)
    position_embeddings[:,1::2] = torch.cos(token_positions/denominator)
    return position_embeddings



In [ ]:
position_embeddings(3,10).unsqueeze(0)

tensor([[[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
           1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  5.4030e-01,  8.7168e-06,  1.0000e+00,  7.5982e-11,
           1.0000e+00,  6.6232e-16,  1.0000e+00,  5.7733e-21,  1.0000e+00],
         [ 9.0930e-01, -4.1615e-01,  1.7434e-05,  1.0000e+00,  1.5196e-10,
           1.0000e+00,  1.3246e-15,  1.0000e+00,  1.1547e-20,  1.0000e+00]]])

# Layer Normalization

In [ ]:
class LayerNormalization(nn.Module):

  def __init__(self,features, eps=1e-6):

    super().__init__()

    gamma = nn.Parameter(torch.zeros(features))
    beta = nn.Parameter(torch.ones(features))

  def forwar(self,x):

    mean = x.mean(-1, keepdim=True)
    std = x.std(-1, keepdim=True)
    return self.gamma*(x-mean)/(std+self.eps) + (self.beta)


# Residual Connection

In [ ]:
class ResidualConnection(nn.Module):

  def __init__(self,features:int, dropout:int):

    super().__init__()
    self.layer_norm = LayerNormalization(features)
    self.dropout = nn.Dropout(dropout)

  def forward(self,x,sublayer):

    return x + self.dropout(sublayer(self.layer_norm(x)))

# MultiHead Attention

In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, d_model, num_heads, dropout_pct):

    super().__init__()
    self.w_q = nn.Linear(d_model, d_model)
    self.w_k = nn.Linear(d_model, d_model)
    self.w_v = nn.Linear(d_model, d_model)
    self.w_o = nn.Linear(d_model, d_model)
    self.dropout = nn.Dropout(dropout_pct)
    self.d_head = d_model//num_heads

    self.num_heads = num_heads

  def forward(self,q, k, v, mask):
    q = self.w_q(q)
    k = self.w_k(k)
    v = self.w_v(v)

    B,T,C= q.shape
    q = q.view(B,T,self.num_heads,self.d_head).transpose(1,2)
    k = k.view(B,T,self.num_heads,self.d_head).transpose(1,2)
    v = v.view(B,T,self.num_heads,self.d_head).transpose(1,2)

    attention = self.calc_attention(q,k, self.d_head, mask)

    attention_scores = attention_scores.transpose(1,2).contiguous().view(B,T,C)
    return self.w_o(attention_scores)

  def calc_attention(self, q, k, v, d_head, mask, dropout):

    attention = q @ k.transpose(-2,-1)
    attention = attention/d_head**0.5
    attention = attention.masked_fill(mask==0, -1e9)
    attention_scores = nn.functional.softmax(attention, dim=-1)
    attention_scores = self.droput(attention_scores)
    attention_scores = attention_scores @ v

    return attention_scores



# Feed Forward NN

In [ ]:
class FeedForward(nn.Module):

  def __init__(self,d_model:int,d_proj:int,dropout_pct:float):

    self.linear_1 = nn.Linear(d_model,d_proj)
    self.dropout = nn.Dropout(dropout_pct)
    self.linear_2 = nn.Linear(d_proj,d_model)


  def forward(self, x):
    return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

# Encoder Block

In [ ]:
class EncoderBlock(nn.Module):

  def __init__(self, attention:MultiHeadAttention , feed_forward:FeedForward , features:int, dropout_pct:float):

      self.attention = attention
      self.feed_forward = feed_forward
      self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout_pct) for _ in range(2)])


  def forward(self, x, mask):

    x = self.residual_connection[0](x, lambda x: self.attention(x,x,x,mask))
    x = self.residual_connections[1](x,self.feed_forward)
    return x


# Encoder Stack

In [ ]:
class EncoderStack(nn.Module):

  def __init__(self, features:int, layers:nn.ModuleList):

    self.layers = layers
    self.norm = LayerNormalization(features)

  def forward(self, x, mask):

    for layer in self.layers:
        x= layer(x,mask)

    return self.norm(x)




# Decoder Block

In [ ]:
class DecoderBlock(nn.Module):

  def __init__(self, attention_block:MultiHeadAttention , cross_attention_block: MultiHeadAttention, feed_forward:FeedForward , features:int, dropout_pct:float):

    self.attention_block = attention_block
    self.cross_attention = cross_attention_block
    self.feed_forward = feed_forward
    self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout_pct) for _ in range(3)])

  def forward(self,x, encoder_output, src_mask,tgt_mask):
    x = self.residual_connections[0](x, lambda x: self.attention_block(x,x,x,tgt_mask))
    x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x,encoder_output,encoder_output,src_mask))
    x = self.residual_connections[2](x,self.feed_forward)
    return x

# Decoder Stack

In [ ]:
class DecoderStack(nn.Module):

  def __init__(self, features:int, layers:nn.ModuleList):

    self.norm = nn.LayerNormalization(features)
    self.layers = layers

  def forward(self,x,encoder_output,input_mask,tgt_mask):
    for layer in self.layers:
      x= layer(x,encoder_output,input_mask,tgt_mask)

    return self.norm(x)

# Linear Projection

In [ ]:
class LinearProjection(nn.Module):

  def __init__(self, d_model, vocab_size):

    super().__init__()
    self.linear_layer = nn.Linear(d_model, vocab_size)

  def forward(self, x):
    return self.linear_layer(x)



# Transfomer

In [ ]:
class Transfomer(nn.Module):

  def __init__(self , encoder:EncoderStack, decoder:DecoderStack , src_embed:InputEmbeddings, tgt_embed:InputEmbeddings, src_pos:PositionalEmbeddings, tgt_pos:PositionalEmbeddings , projection:LinearProjection)

    super().__init__()
    self.encoder  = encoder
    self.decoder  = decoder
    self.src_embed = src_embed
    self.tgt_embed = tgt_embed
    self.src_pos = src_pos
    self.tgt_pos = tgt_pos
    self.projection = projection

  def encode(self, x, src_mask ):

    x= self.src_embed(x)
    x= self.src_pos(x)
    x= self.encoder(x, src_mask)
    return x

  def decode(self,x,encoder_output,src_mask,tgt_mask):

    x = self.tgt_embed(x)
    x = self.tgt_pos(x)
    x = self.decoder(x,encoder_output,src_mask,tgt_mask)
    return x

  def projection(self,x):

     return self.projection(x)



In [ ]:
def build_transformer(input_vocabsize:int, target_vocabsize:int, input_seqlen:int, target_seqlen:int, d_model:int, num_heads:int, num_layers:int, d_ff:int, dropout_pct:float ):

    src_embeddings = InputEmbeddings(d_model, input_vocabsize)
    tgt_embeddings = InputEmbeddings(d_model, target_vocabsize)

    src_pos = PositionalEmbeddings(input_seqlen, d_model)
    tgt_pos = PositionalEmbeddings(target_seqlen, d_model)


    encoder_blocks = []
    decoder_blocks = []

    for layer in num_layers:
      attention = MultiHeadAttention(d_model, num_heads, dropout_pct)
      feed_forward = FeedForward(d_model, d_ff, dropout_pct)
      encoder_block = EncoderBlock(attention, feed_forward, d_model, dropout_pct)
      encoder_block.append(encoder_block)

    for layer in num_layers:
      attention = MultiHeadAttention(d_model,num_heads,dropout_pct)
      cross_attention = MultiHeadAttention(d_model, num_heads, dropout_pct)
      feed_forward = FeedForward(d_model, d_ff, dropout_pct)
      decoder_block = DecoderBlock(attention, cross_attention, feed_forward, d_model, dropout_pct)
      decoder_blocks.append(decoder_block)


    encoder  = EncoderStack(d_model,nn.ModuleList(encoder_blocks))
    decoder  = DecoderStack(d_model,nn.ModuleList(decoder_blocks))

    projection = LinearProjection(d_model, target_vocabsize)

    transformer = Transfomer(encoder, decoder, src_embeddings, tgt_embeddings, src_pos, tgt_pos, projection)

    return transformer



